# DiD-BCF — B1_strong_confounder (linearity_degree=1)

**Workstream B1 · canonical DiD (selection on unobservables)**

large Var(alpha) and strong corr(alpha, treatment)

Fits DiD-BCF on the `B1_strong_confounder` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_strong_confounder",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_strong_confounder_lin_1] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_strong_confounder: 100%|██████████| 100/100 [4:20:41<00:00, 156.42s/fit]

[B1_strong_confounder_lin_1] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_strong_confounder_lin_1.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_strong_confounder,1,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.258000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
1,canonical,B1_strong_confounder,1,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.246000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
2,canonical,B1_strong_confounder,1,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.427333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
3,canonical,B1_strong_confounder,1,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.457333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
4,canonical,B1_strong_confounder,1,200,0,ES,k=0,NaN,NaN,0.0,...,0.258000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_strong_confounder,1,200,ATT,ATT,power,-1.119238,-3.757694,0.13,...,0.768675,1.726120,1.160536,3.757694,1.0,0.0,1.363075,3.759757,0.255831,5.527904
1,canonical,B1_strong_confounder,1,200,ES,k=0,power,-1.124097,-3.758033,0.18,...,0.939405,1.589923,1.160034,3.758033,1.0,0.0,1.363831,3.759134,0.314045,31.980259
2,canonical,B1_strong_confounder,1,200,ES,k=1,power,-1.109194,-3.757281,0.15,...,0.952251,1.783086,1.160049,3.757281,1.0,0.0,1.362692,3.759737,0.310670,4.687149
3,canonical,B1_strong_confounder,1,200,ES,k=2,power,-1.115680,-3.756592,0.17,...,0.938991,1.904815,1.169567,3.756592,1.0,0.0,1.376154,3.759388,0.302160,4.494060
4,canonical,B1_strong_confounder,1,200,ES,k=3,power,-1.127980,-3.758870,0.16,...,0.932769,2.019946,1.163293,3.758870,1.0,0.0,1.364893,3.762357,0.314207,3.973312
5,canonical,B1_strong_confounder,1,200,GATT,g=4_t=4,power,-1.124097,-3.758033,0.18,...,0.939405,1.589923,1.160034,3.758033,1.0,0.0,1.363831,3.759134,0.314045,31.980259
6,canonical,B1_strong_confounder,1,200,GATT,g=4_t=5,power,-1.109194,-3.757281,0.15,...,0.952251,1.783086,1.160049,3.757281,1.0,0.0,1.362692,3.759737,0.310670,4.687149
7,canonical,B1_strong_confounder,1,200,GATT,g=4_t=6,power,-1.115680,-3.756592,0.17,...,0.938991,1.904815,1.169567,3.756592,1.0,0.0,1.376154,3.759388,0.302160,4.494060
8,canonical,B1_strong_confounder,1,200,GATT,g=4_t=7,power,-1.127980,-3.758870,0.16,...,0.932769,2.019946,1.163293,3.758870,1.0,0.0,1.364893,3.762357,0.314207,3.973312


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_strong_confounder,1,200,corrected,100,401.76,1.526439,0.554454,1.331315,...,0.335314,0.137112,0.173242,0.117553,0.208399,0.134208,0.783794,0.106195,0.940854,0.134075
1,canonical,B1_strong_confounder,1,200,plain,100,401.76,3.858635,0.123259,3.757694,...,0.999050,0.020495,0.000000,0.000000,0.000000,0.000000,1.449776,0.082997,1.831239,0.098646
